# Notebook 4 — Evaluation

This notebook carries out the full quantitative and qualitative evaluation
of the two FER2013 classifiers trained in Notebook 3. No training occurs
here: we load the saved models and histories from disk and treat them as
fixed artefacts.

What will be produced:

1. **Training curves** — loss and accuracy over epochs, overfitting diagnosis.
2. **Confusion matrices** — normalised heatmaps revealing per-class error patterns.
3. **Classification report** — precision, recall, F1 and support per emotion class.
4. **Comparative summary table** — accuracy, F1, inference latency, parameter count.
5. **Error analysis** — image grids for the most frequent misclassification pairs.
6. **Grad-CAM++ visualisation** — which facial regions drive each emotion prediction.
7. **Receptivity index simulation** — 200-frame synthetic sales call with index trajectory.
8. **Final model selection** — weighted scoring with explicit winner justification.
9. **Critical reflection** — limitations, bias, and future improvements.

## Section 1 — Setup

We load both trained models and their histories from disk; no training
happens in this notebook. We also load the held-out test set from the
preprocessed `.npz` files produced in Notebook 2. Using persisted files
guarantees that evaluation is performed on exactly the same split and
normalisation as training — recomputing preprocessing here would risk
subtle inconsistencies (e.g. a different random seed in the train/val split).

In [ ]:
import json
import sys
import time
from collections import Counter
from pathlib import Path

import matplotlib.cm as mpl_cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.inference.receptivity_mapper import ReceptivityIndex

# ── Test data ──────────────────────────────────────────────────────────────
gray = np.load(config.PROCESSED_DIR / "fer2013_gray.npz")
rgb  = np.load(config.PROCESSED_DIR / "fer2013_rgb64.npz")

X_test_gray = gray["X_test"]
X_test_rgb  = rgb["X_test"]
y_test_oh   = gray["y_test"]
y_test      = np.argmax(y_test_oh, axis=1)

# ── Models ─────────────────────────────────────────────────────────────────
def _load_model(name):
    path = config.MODELS_DIR / f"{name}.keras"
    if path.exists():
        return keras.models.load_model(str(path)), path
    path = config.MODELS_DIR / f"{name}.h5"
    return keras.models.load_model(str(path)), path

model_cnn, path_cnn = _load_model("cnn_custom")
model_mob, path_mob = _load_model("mobilenet_ft")

# ── Histories ──────────────────────────────────────────────────────────────
with open(config.HISTORIES_DIR / "cnn_custom_history.json") as f:
    hist_cnn = json.load(f)
with open(config.HISTORIES_DIR / "mobilenet_ft_history.json") as f:
    hist_mob = json.load(f)

LABELS = config.EMOTION_LABELS

print(f"Custom CNN    loaded from: {path_cnn}")
print(f"MobileNetV2   loaded from: {path_mob}")
print(f"Test set      gray={X_test_gray.shape}  rgb={X_test_rgb.shape}  y={y_test.shape}")
print(f"CNN history   {len(hist_cnn['loss'])} epochs")
print(f"MobileNet history  {len(hist_mob['loss'])} epochs")

## Section 2 — Training curves

Loss and accuracy are plotted for both the training and validation sets
across all epochs. A dashed vertical line marks the epoch restored by
`EarlyStopping` — i.e. the epoch with the lowest validation loss. A large
gap between training and validation curves at that point indicates
overfitting; curves that never converge indicate underfitting. For the
MobileNetV2 combined history the Stage 1 / Stage 2 boundary is also visible
as a drop in learning rate around epoch 10.

In [ ]:
def _plot_curves(axes_row, history, title):
    epochs = range(1, len(history["loss"]) + 1)
    best_ep = int(np.argmin(history["val_loss"])) + 1

    ax_l, ax_a = axes_row
    ax_l.plot(epochs, history["loss"],     label="Train")
    ax_l.plot(epochs, history["val_loss"],  label="Val")
    ax_l.axvline(best_ep, color="gray", linestyle="--", alpha=0.6, label=f"Best ep {best_ep}")
    ax_l.set_title(f"{title} — Loss"); ax_l.set_xlabel("Epoch")
    ax_l.legend(); ax_l.grid(True, alpha=0.3)

    ax_a.plot(epochs, history["accuracy"],     label="Train")
    ax_a.plot(epochs, history["val_accuracy"],  label="Val")
    ax_a.axvline(best_ep, color="gray", linestyle="--", alpha=0.6)
    ax_a.set_title(f"{title} — Accuracy"); ax_a.set_xlabel("Epoch")
    ax_a.legend(); ax_a.grid(True, alpha=0.3)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
_plot_curves(axes[0], hist_cnn, "Custom CNN")
_plot_curves(axes[1], hist_mob, "MobileNetV2 (Stage 1 + Stage 2)")
plt.tight_layout()
plt.show()

## Section 3 — Test set predictions

We run a single inference pass over the full test set for each model and
store the resulting class probabilities and argmax labels. All downstream
analyses (confusion matrices, classification reports, error grids, Grad-CAM
sample selection) reuse these arrays without calling `model.predict` again.
This ensures that every metric in the notebook refers to the same prediction
run and that the test set is not touched more than once.

In [ ]:
prob_cnn = model_cnn.predict(X_test_gray, verbose=0)   # (n, 7)
prob_mob = model_mob.predict(X_test_rgb,  verbose=0)   # (n, 7)

pred_cnn = np.argmax(prob_cnn, axis=1)
pred_mob = np.argmax(prob_mob, axis=1)

acc_cnn = float((pred_cnn == y_test).mean())
acc_mob = float((pred_mob == y_test).mean())

print(f"Custom CNN  — test accuracy: {acc_cnn:.4f}")
print(f"MobileNetV2 — test accuracy: {acc_mob:.4f}")